<p style="text-align:center">
    <a href="https://tukkalearn.vercel.app" target="_blank">
    <img src="https://raw.githubusercontent.com/itzDM/publicAssets/refs/heads/main/opengraph-image.png" width="250"  alt="Tukka Learn">
    </a>
</p>


In [16]:
import pandas as pd
import numpy as np

In [17]:
df=pd.read_csv('https://raw.githubusercontent.com/tukkaLearn/datasets/refs/heads/main/ecommerce_nov2025_messy.csv')
df.head()

,order_id,customer_id,order_date,product_name,category,price,quantity,discount,payment_method,status,email,country,signup_date
0,ORD1001,CUST_901,2025-11-01,iPhone 16 Pro,Electronics,1299,1,0.10,Credit Card,Delivered,john.doe@gmail.com,USA,2023-05-12
1,ORD1002,cust_902,01/11/2025,AirPods Pro 2,electronics,249,2,NaN,PayPal,Pending,NaN,Canada,NaN
2,ORD1003,CUST_903,Nov 02 2025,Samsung Galaxy S25,Electronics,1199,1,0.00,credit card,Cancelled,alice_23@yahoo.com,UK,2024-01-30
3,ORD1004,CUST_901,2025-11-03,MacBook Air M3,Electronics,$1099,1,0.15,Apple Pay,Delivered,john.doe@gmail.com,USA,2023-05-12
4,ORD1005,CUST_905,2025-11-03,T-Shirt Oversized,Fashion,29.99,5,NaN,Cash on Delivery,Delivered,bob.builder@hotmail.com,India,2025-10-28


### Convert string to proper datetime


In [18]:
df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce', dayfirst=True)
df['signup_date'] = pd.to_datetime(df['signup_date'], errors='coerce')

print(f"Failed dates → {df['order_date'].isna().sum()} (now NaT)")
df[['order_id', 'order_date', 'signup_date']].head(10)

Failed dates → 6 (now NaT)


,order_id,order_date,signup_date
0,ORD1001,2025-01-11,2023-05-12
1,ORD1002,NaT,NaT
2,ORD1003,NaT,2024-01-30
3,ORD1004,2025-03-11,2023-05-12
4,ORD1005,2025-03-11,2025-10-28
5,ORD1006,2025-04-11,2024-08-15
6,ORD1007,NaT,2025-01-10
7,ORD1008,2025-05-11,2023-05-12
8,ORD1009,2025-05-11,2023-12-01
9,ORD1010,NaT,NaT


### Extract everything with .dt


In [20]:
df['date_only']      = df['order_date'].dt.date
df['hour']           = df['order_date'].dt.hour
df['day_name']       = df['order_date'].dt.day_name()
df['is_weekend']     = df['order_date'].dt.weekday >= 5
df['week_number']    = df['order_date'].dt.weekday        # 0=Mon
df['month_end']      = df['order_date'].dt.is_month_end

df[['order_id', 'order_date', 'date_only', 'hour', 'day_name', 'is_weekend', 'week_number', 'month_end']].head()

,order_id,order_date,date_only,hour,day_name,is_weekend,week_number,month_end
0,ORD1001,2025-01-11,2025-01-11,0.0,Saturday,True,5.0,False
1,ORD1002,NaT,NaT,NaN,NaN,False,NaN,False
2,ORD1003,NaT,NaT,NaN,NaN,False,NaN,False
3,ORD1004,2025-03-11,2025-03-11,0.0,Tuesday,False,1.0,False
4,ORD1005,2025-03-11,2025-03-11,0.0,Tuesday,False,1.0,False


### Time differences – customer journey


In [ ]:
# Calculate minutes between order_created and order_delivered.
# Create column delivery_time_minutes → show average and 95th percentile.


In [21]:
df['days_since_signup'] = (df['order_date'] - df['signup_date']).dt.days
df['churn_risk'] = df['days_since_signup'] > 180

print(f"Customers at risk (no order in 6+ months): {df['churn_risk'].sum():,} ({df['churn_risk'].mean():.1%})")
df[['customer_id', 'signup_date', 'order_date', 'days_since_signup', 'churn_risk']].head()

Customers at risk (no order in 6+ months): 6 (40.0%)


,customer_id,signup_date,order_date,days_since_signup,churn_risk
0,CUST_901,2023-05-12,2025-01-11,610.0,True
1,cust_902,NaT,NaT,NaN,False
2,CUST_903,2024-01-30,NaT,NaN,False
3,CUST_901,2023-05-12,2025-03-11,669.0,True
4,CUST_905,2025-10-28,2025-03-11,-231.0,False


### Exercise 4: Black Friday week filter


In [ ]:
bf_start = '2025-11-24'
bf_end   = '2025-12-01'

bf_sales = df[df['order_date'].between(bf_start, bf_end)]
print(f"Black Friday week revenue: ${bf_sales['total_amount'].sum():,.0f}")
print(f"Normal week avg: ${df[df['order_date'].dt.month == 11]['total_amount'].sum() / 4:,.0f}")
print(f"→ {bf_sales['total_amount'].sum() / (df[df['order_date'].dt.month == 11]['total_amount'].sum() / 4):.1f}x higher!")

KeyError: 'total_amount'

### Exercise 5: Generate missing dates (hourly sales)


In [ ]:
# Simulate hourly data
hourly = df.set_index('order_date').resample('H')['total_amount'].sum().reset_index()

# Create perfect hourly index for November
full_hours = pd.date_range('2025-11-01', '2025-11-30 23:59:59', freq='H')
hourly_full = hourly.set_index('order_date').reindex(full_hours).fillna(0).reset_index()
hourly_full = hourly_full.rename(columns={'index': 'hour'})

print(f"Missing hours before reindex: {hourly.shape[0]}")
print(f"After perfect grid: {hourly_full.shape[0]} hours → {hourly_full['total_amount'].eq(0).sum()} quiet hours")
hourly_full.tail()

### Exercise 6: Resample to daily, weekly, monthly


In [ ]:
daily  = df.set_index('order_date').resample('D')['total_amount'].sum()
weekly = df.set_index('order_date').resample('W-MON')['total_amount'].sum()
monthly = df.set_index('order_date').resample('M')['total_amount'].sum()

plt.figure(figsize=(15,8))
daily.plot(alpha=0.6, label='Daily')
daily.rolling(7).mean().plot(linewidth=3, label='7-day MA')
weekly.plot(marker='o', linewidth=3, label='Weekly')
plt.title('Revenue Over Time – Multiple Frequencies')
plt.legend()
plt.show()

### Exercise 7: Peak shopping hour


In [ ]:
hourly_revenue = df.groupby(df['order_date'].dt.hour)['total_amount'].mean()
peak_hour = hourly_revenue.idxmax()

hourly_revenue.plot(kind='bar', figsize=(10,5), color='skyblue')
plt.title(f'Peak Shopping Hour = {peak_hour}:00–{peak_hour+1}:00')
plt.xlabel('Hour of Day')
plt.ylabel('Avg Revenue per Order')
plt.show()

### Exercise 8: Weekend vs Weekday performance


In [ ]:
weekend = df.groupby('is_weekend')['total_amount'].agg(['sum', 'mean', 'count'])
weekend['revenue_pct'] = weekend['sum'] / weekend['sum'].sum() * 100
weekend.round(0)

### Exercise 9: Timezone conversion (USA to India)


In [ ]:
df_usa = df.copy()
df_usa['order_date_est'] = df_usa['order_date'].dt.tz_localize('UTC').dt.tz_convert('US/Eastern')
df_usa['order_date_ist'] = df_usa['order_date'].dt.tz_localize('UTC').dt.tz_convert('Asia/Kolkata')

print("Same order in different timezones:")
df_usa[['order_date', 'order_date_est', 'order_date_ist']].head(3)

### Exercise 10: Rolling 7-day revenue


In [ ]:
df_daily = df.set_index('order_date').sort_index()
df_daily['revenue_7d'] = df_daily['total_amount'].rolling('7D').sum()
df_daily['revenue_7d_ma'] = df_daily['total_amount'].rolling(window=7).mean()

df_daily[['total_amount', 'revenue_7d_ma']].plot(figsize=(15,5), alpha=0.8)
plt.title('7-Day Rolling Average Revenue')
plt.show()

### Exercise 11: Shift – revenue vs yesterday


In [ ]:
daily_shift = daily.to_frame()
daily_shift['yesterday'] = daily_shift['total_amount'].shift(1)
daily_shift['growth'] = daily_shift['total_amount'] / daily_shift['yesterday'] - 1

print("Top 5 growth days:")
daily_shift.nlargest(5, 'growth')

### Exercise 12: Longest quiet period (zero sales)


In [ ]:
gaps = daily[daily == 0].index.to_series().diff().dt.days
longest_gap = gaps.max()

print(f"Longest period with zero sales: {longest_gap} days")

### Exercise 13: FINAL BOSS – One-function time-series dashboard


In [ ]:
def time_series_dashboard(df):
    df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
    ts = df.set_index('order_date').sort_index()['total_amount']
    daily = ts.resample('D').sum()
    weekly = ts.resample('W-MON').sum()
    
    print("TIME-SERIES DASHBOARD – NOV 2025")
    print("="*50)
    print(f"Total Revenue        : ${ts.sum():,.0f}")
    print(f"Peak Day             : {daily.idxmax().date()} → ${daily.max():,.0f}")
    print(f"Peak Hour            : {df['order_date'].dt.hour.mode().iloc[0]}:00")
    print(f"Weekend % of Revenue : {df[df['is_weekend']]['total_amount'].sum() / ts.sum():.1%}")
    
    daily.rolling(7).mean().plot(figsize=(15,6), linewidth=3, label='7-day MA')
    weekly.plot(marker='o', linewidth=4, label='Weekly Total')
    plt.axvline('2025-11-28', color='red', linestyle='--', linewidth=2, label='Black Friday')
    plt.title('Revenue Trend – Daily (smoothed) + Weekly')
    plt.legend()
    plt.show()

time_series_dashboard(df)

## YOU ARE NOW A TIME-SERIES NINJA

You can handle any date mess, any timezone, any frequency.
You are the person everyone calls when revenue numbers don’t match.

**Next elite notebooks ready:**

- `give me merge & concat notebook` (the merge hell conqueror)
- `give me pivot & reshape notebook`
- `give me full course zip` (all 10 notebooks + datasets)

Just say **"next"** or pick one.

You are officially unstoppable.
